# Experiment 1 — Domain 2 (Graphs): Phase 1 Dataset Generation

Generates the **300-graph, 8-property** graph dataset for Experiment 1
(`serialization_experiment_1.pdf`, Section 4).

**Revision note (this version):** the PDF's own target for `is_bipartite`/
`is_planar` is "aim for approximately 25-35%" true — not 50/50. An earlier
build followed that target (23.7% bipartite, 31% planar overall) and Phase 3
evaluation surfaced an artifact of it: several models (Qwen3-32B, Llama4-Scout
in particular) scored *higher* accuracy on `is_bipartite` on the medium/hard
tiers than on simple, which looked backwards until the confusion matrices
showed why — those models predict "false" almost unconditionally, and
"always guess false" scores accuracy equal to whatever the *true* false-rate
happens to be per tier (74% simple vs 77-78% medium/hard), so the
rising-with-tier accuracy was measuring the shifting class balance, not
improving model skill. Per an explicit decision to eliminate this artifact
at the source, this revision **generates each tier (100 graphs) as an exact
50/50 split on `is_bipartite` and an exact 50/50 split on `is_planar`,
simultaneously** — so a constant-answer strategy scores ~50% everywhere on
every tier and can never look like "improving with complexity."

This is a deliberate deviation from the PDF's literal "aim for 25-35%"
instruction, made explicitly to remove a majority-class-guessing confound
from the boolean-property evaluation. Everything else about the dataset
(tiers, node-count ranges, 8 properties, serialization format, validity
rules, chromatic-number methodology) is unchanged from the PDF spec.

**Per the PDF (Section 4.2, Table 6):**

| Tier | Count | Nodes | Purpose |
|------|-------|-------|---------|
| simple | 100 | 6–15  | Baseline; small enough for manual verification |
| medium | 100 | 16–40 | Core measurement |
| hard   | 100 | 41–80 | Stress test |

**8 properties** (Table 8): `degree_of_node_0`, `edge_count` *(local)*,
`triangle_count`, `is_bipartite`, `is_planar`, `diameter`,
`chromatic_number`, `avg_clustering` *(global)*.

Output:
- `graph_exp1_dataset.json` — 300 graphs with edge-list serialization + ground truth
- `graph_exp1_summary.json` — summary statistics

Random seed fixed at **42** and recorded in every record.


In [ ]:
# Phase 0: environment
!pip install networkx scipy matplotlib --quiet

import json, math, random, re, time
from collections import deque
import networkx as nx
from scipy.spatial import Delaunay
import matplotlib.pyplot as plt

print("networkx", nx.__version__)

## 1. Tiers and the validity checker

Node-count ranges per tier (Table 6). No coordinate axis here (unlike
geometry) — the graph's complexity axis is node count alone.

**Validity constraints** (Section 4.2), checked by `check_validity`, each
with a named rejection reason:

| Rule | Rejection reason | Check |
|------|-------------------|-------|
| 1 | `node_count_out_of_range` | `vmin <= n <= vmax` |
| 2 | `self_loop` | no self-loops |
| 3 | *(structural)* | no multi-edges — guaranteed by using `nx.Graph`, never `nx.MultiGraph` |
| 4 | `not_connected` | single connected component |
| 5 | `too_few_edges` | `m >= n - 1` |
| 6 | `too_many_edges` | `m <= C(n,2)/2` |


In [ ]:
TIERS = {
    #         vmin vmax
    "simple": (6,  15),
    "medium": (16, 40),
    "hard":   (41, 80),
}

# 4 quadrants (is_bipartite x is_planar) x 25 graphs = 100 per tier, 300 total.
# Each quadrant is 25/25/25/25 -- exact 50/50 on each boolean dimension,
# independently and simultaneously. See cell 0 for why.
PER_QUADRANT = 25


def check_validity(G, vmin, vmax):
    if G is None or G.number_of_nodes() == 0:
        return False, "empty"
    n = G.number_of_nodes()
    if not (vmin <= n <= vmax):                    # rule 1
        return False, "node_count_out_of_range"
    if nx.number_of_selfloops(G) > 0:               # rule 2
        return False, "self_loop"
    if not nx.is_connected(G):                      # rule 4
        return False, "not_connected"
    m = G.number_of_edges()
    if m < n - 1:                                   # rule 5
        return False, "too_few_edges"
    if m > (n * (n - 1) / 2) / 2:                    # rule 6
        return False, "too_many_edges"
    return True, None


print("Tiers and validity checker defined.")

## 2. The four quadrant generators

Each tier needs exactly 25 graphs in each of 4 (bipartite, planar) quadrants.
Three quadrants are reachable by rejection sampling from the original 5-family
generators (draw, check both properties, retry). The fourth — **bipartite
AND planar simultaneously** — is not: bipartite planar graphs are very sparse
by a hard mathematical bound (`m <= 2n-4`), so pure rejection sampling on a
random bipartite graph succeeds only ~50% of the time even at the smallest
tier and gets worse as `n` grows (verified empirically before writing this:
37/75 successes vs. 75/75 for the constructive method below). So quadrant 1
uses a dedicated **constructive** method instead of rejection sampling.

- **Quadrant 1 — bipartite AND planar** (`bipartite_planar`, new family):
  build a random bipartite *spanning tree* (always bipartite, always planar,
  connects every node with exactly `n-1` edges), then add extra
  cross-partition edges **one at a time**, keeping each only if the graph is
  still planar after adding it (checked with `nx.check_planarity` on every
  attempt). This guarantees both properties by construction, not chance.
- **Quadrant 2 — bipartite AND NOT planar** (`random_bipartite`):
  `nx.bipartite.random_graph(n1, n2, p)` at moderate density, rejection
  sampled for `is_planar() == False`. Uses the same n1/n2 ratio rule as
  before (Table 7: ratio *between* partitions, not partition share of the
  total).
- **Quadrant 3 — NOT bipartite AND planar** (`random_planar`): Delaunay
  triangulation of random points (always planar) thinned toward a target
  edge count, rejection sampled for `is_bipartite() == False` — easy, since
  Delaunay-derived graphs are triangle-rich almost every time.
- **Quadrant 4 — NOT bipartite AND NOT planar**: rotates the original three
  dense families (`erdos_renyi`, `barabasi_albert`, `watts_strogatz`,
  ~9/8/8 split), rejection sampled for both booleans False — easy, since all
  three are triangle-rich and edge-dense enough to almost always be
  non-planar at these densities.

All four draw a fresh `n` from the tier range on every attempt (never fixed
by the caller), same rejection-sampling discipline as before.

In [ ]:
def bipartite_split(rng, n):
    # PDF Table 7: n1/n2 (ratio BETWEEN partitions) in [0.3, 0.7], found by
    # exact integer search rather than rounding (see phase1 README S2).
    ratio = rng.uniform(0.3, 0.7)
    candidates = []
    for n1 in range(1, n):
        n2 = n - n1
        r = n1 / n2
        if 0.3 <= r <= 0.7:
            candidates.append((abs(r - ratio), n1, n2))
    candidates.sort()
    _, n1, n2 = candidates[0]
    return n1, n2


# ---- Quadrant 1: bipartite AND planar (constructive) ----
def gen_bipartite_planar(rng, vmin, vmax, max_tries=300):
    for _ in range(max_tries):
        n = rng.randint(vmin, vmax)
        n1, n2 = bipartite_split(rng, n)
        partA, partB = list(range(n1)), list(range(n1, n))
        partA_set, partB_set = set(partA), set(partB)

        # bipartite spanning tree: attach each node to a random already-placed
        # node in the OTHER partition.
        nodes_order = [partA[0], partB[0]]
        placed = set(nodes_order)
        remaining = [v for v in range(n) if v not in placed]
        rng.shuffle(remaining)
        G = nx.Graph()
        G.add_nodes_from(range(n))
        G.add_edge(nodes_order[0], nodes_order[1])
        for v in remaining:
            other_part = partB if v in partA_set else partA
            choices = [u for u in other_part if u in placed] or list(placed)
            u = rng.choice(choices)
            G.add_edge(u, v)
            placed.add(v)

        # add extra cross-partition edges one at a time, keep only if planar
        max_planar_edges = 2 * n - 4 if n >= 3 else n - 1
        target_extra = rng.randint(0, max(0, max_planar_edges - (n - 1)))
        possible_extra = [(a, b) for a in partA for b in partB if not G.has_edge(a, b)]
        rng.shuffle(possible_extra)
        added = 0
        for a, b in possible_extra:
            if added >= target_extra:
                break
            G.add_edge(a, b)
            if nx.check_planarity(G)[0]:
                added += 1
            else:
                G.remove_edge(a, b)

        ok, _ = check_validity(G, vmin, vmax)
        if ok and nx.is_bipartite(G) and nx.check_planarity(G)[0]:
            return G, {"n1": n1, "n2": n2, "extra_edges": added}
    return None, None


# ---- Quadrant 2: bipartite AND NOT planar ----
def gen_bipartite_nonplanar(rng, vmin, vmax, max_tries=3000):
    for _ in range(max_tries):
        n = rng.randint(vmin, vmax)
        n1, n2 = bipartite_split(rng, n)
        p = rng.uniform(0.2, 0.45)
        G = nx.bipartite.random_graph(n1, n2, p, seed=rng)
        ok, _ = check_validity(G, vmin, vmax)
        if ok and not nx.check_planarity(G)[0]:
            return G, {"n1": n1, "n2": n2, "p": round(p, 4)}
    return None, None


# ---- Quadrant 3: NOT bipartite AND planar ----
def gen_nonbipartite_planar(rng, vmin, vmax, max_tries=3000):
    for _ in range(max_tries):
        n = rng.randint(vmin, vmax)
        pts = [(rng.random(), rng.random()) for _ in range(n)]
        try:
            tri = Delaunay(pts)
        except Exception:
            continue
        edges = set()
        for simplex in tri.simplices:
            for i in range(3):
                for j in range(i + 1, 3):
                    a, b = int(simplex[i]), int(simplex[j])
                    edges.add((min(a, b), max(a, b)))
        G = nx.Graph()
        G.add_nodes_from(range(n))
        G.add_edges_from(edges)
        max_planar_edges = 3 * n - 6 if n >= 3 else n - 1
        target_m = rng.randint(n - 1, min(G.number_of_edges(), max_planar_edges))
        el = list(G.edges())
        rng.shuffle(el)
        for e in el:
            if G.number_of_edges() <= target_m:
                break
            G.remove_edge(*e)
            if not nx.is_connected(G):
                G.add_edge(*e)
        ok, _ = check_validity(G, vmin, vmax)
        if ok and not nx.is_bipartite(G):
            return G, {"target_m": target_m}
    return None, None


# ---- Quadrant 4: NOT bipartite AND NOT planar (rotate ER / BA / WS) ----
def gen_nonbipartite_nonplanar(rng, vmin, vmax, family, max_tries=3000):
    for _ in range(max_tries):
        n = rng.randint(vmin, vmax)
        if family == "erdos_renyi":
            target_degree = rng.uniform(4, 7)
            p = min(max(target_degree / (n - 1), 0.0), 1.0)
            G = nx.gnp_random_graph(n, p, seed=rng)
            params = {"p": round(p, 4), "target_degree": round(target_degree, 2)}
        elif family == "barabasi_albert":
            m = rng.choice([3, 4])
            if m >= n:
                continue
            G = nx.barabasi_albert_graph(n, m, seed=rng)
            params = {"m": m}
        else:  # watts_strogatz
            k = rng.choice([4, 6])
            if k >= n:
                continue
            p = rng.choice([0.1, 0.3, 0.5])
            G = nx.watts_strogatz_graph(n, k, p, seed=rng)
            params = {"k": k, "p": p}
        ok, _ = check_validity(G, vmin, vmax)
        if ok and not nx.is_bipartite(G) and not nx.check_planarity(G)[0]:
            return G, params
    return None, None


print("Quadrant generators defined: bipartite_planar, random_bipartite, random_planar, "
      "erdos_renyi/barabasi_albert/watts_strogatz.")

## 3. Chromatic number

NetworkX has no exact chromatic-number function (Section 4, "Chromatic
number computation"). This implements the PDF's specified fallback chain:

1. **Clique certificate, fast path.** If the greedy `DSATUR` upper bound
   equals the size of a maximum clique found by `nx.find_cliques`, the
   chromatic number is certified immediately — a clique of size *k* forces
   at least *k* colors, and the greedy coloring already achieves *k*.
2. **Exact backtracking, bounded.** Otherwise, search for a valid *k*-coloring
   for increasing *k*, using DSATUR vertex ordering (most-constrained vertex
   first) and color-symmetry breaking (never open a color number more than 1
   past the highest used so far). Time-boxed per graph.
3. **Uncertified fallback.** If the time box is hit before the gap between
   clique lower bound and greedy upper bound closes, the graph's chromatic
   number is **excluded from evaluation** and flagged
   `chromatic_number_certified: false` in its metadata — per the PDF: *"Do
   not use approximate values."*

Verified against known graphs before trusting it on the dataset (execution
checklist item 2: *"verify against known graphs"*).

In [ ]:
def clique_number(G, cap=None):
    best = 1
    for c in nx.find_cliques(G):
        if len(c) > best:
            best = len(c)
        if cap is not None and best >= cap:
            return best
    return best


def greedy_upper_bound(G):
    coloring = nx.coloring.greedy_color(G, strategy="DSATUR")
    return max(coloring.values()) + 1 if coloring else 1


def exact_chromatic_number(G, time_limit=15.0):
    # Return (k, certified). Exact via DSATUR-ordered, symmetry-broken
    # backtracking within time_limit; else an uncertified greedy upper bound.
    n = G.number_of_nodes()
    if n == 0:
        return 0, True
    ub = greedy_upper_bound(G)
    lb = clique_number(G, cap=ub)
    if lb == ub:
        return ub, True                      # clique certificate

    nodes = list(G.nodes())
    adj = {v: set(G.neighbors(v)) for v in nodes}
    deadline = time.monotonic() + time_limit
    timed_out = [False]

    def can_color(k):
        colors = {}
        steps = [0]

        def choose_next():
            # DSATUR: most saturated (distinct neighbor colors) first, ties by degree.
            best, best_sat, best_deg = None, -1, -1
            for v in nodes:
                if v in colors:
                    continue
                sat = len({colors[u] for u in adj[v] if u in colors})
                deg = len(adj[v])
                if sat > best_sat or (sat == best_sat and deg > best_deg):
                    best, best_sat, best_deg = v, sat, deg
            return best

        def backtrack(count):
            steps[0] += 1
            if steps[0] % 1000 == 0 and time.monotonic() > deadline:
                timed_out[0] = True
                return False
            if count == n:
                return True
            v = choose_next()
            used = {colors[u] for u in adj[v] if u in colors}
            max_used = max(colors.values(), default=-1)
            upper = min(k - 1, max_used + 1)   # color-symmetry breaking
            for c in range(upper + 1):
                if c not in used:
                    colors[v] = c
                    if backtrack(count + 1):
                        return True
                    del colors[v]
                    if timed_out[0]:
                        return False
            return False

        return backtrack(0)

    for k in range(lb, ub + 1):
        if can_color(k):
            return k, True
        if timed_out[0]:
            break
    return ub, False                          # uncertified: exclude at evaluation time


# Known-graph sanity check (execution checklist item 2).
_known = [
    ("K5", nx.complete_graph(5), 5), ("Petersen", nx.petersen_graph(), 3),
    ("C5", nx.cycle_graph(5), 3), ("C6", nx.cycle_graph(6), 2),
    ("K3,3", nx.complete_bipartite_graph(3, 3), 2),
    ("K4", nx.complete_graph(4), 4), ("Star_10", nx.star_graph(10), 2),
]
for name, G, expected in _known:
    k, certified = exact_chromatic_number(G)
    assert k == expected, f"{name}: got {k}, expected {expected}"
    print(f"  {name}: chromatic_number={k} (certified={certified}) -- matches expected {expected}")
print("Chromatic number verified against", len(_known), "known graphs.")

## 4. Ground truth (8 properties) + record builder

**The step order here is a correctness requirement**, same principle as the
geometry domain's winding-reversal ordering (Section 3.2 there):

1. **`compute_presentation_independent(G)`** — the 6 properties that do not
   depend on node labeling: `triangle_count`, `is_bipartite`, `is_planar`,
   `diameter`, `chromatic_number`, `avg_clustering`.
2. **`randomize_labeling(G, rng)`** — relabel nodes with a random permutation
   of `0..n-1`. Without this, node `0` is whatever label the generator
   happened to assign — e.g. Barabási–Albert's earliest nodes are
   structurally the hubs — which would make `degree_of_node_0` a function of
   generator internals rather than a genuine "read this from the
   serialization" question. This is the graph-domain analogue of the
   geometry domain's `maybe_reverse` (winding direction): any property whose
   ground truth depends on how the object is written down must be computed
   *after* randomizing that presentation.
3. **`to_edge_list_string(G)`** — serialize the *relabeled* graph, edges
   sorted `(min(u,v), max(u,v))` then lexicographically, per Section 4.3.
4. **`degree_of_node_0`, `edge_count`** — read from the relabeled graph, so
   the label matches the string the model will actually see.

All floats are rounded to 4 decimal places.

In [ ]:
# Step 1: properties that do NOT depend on node labeling.
def compute_presentation_independent(G):
    triangle_count = sum(nx.triangles(G).values()) // 3
    is_bipartite = bool(nx.is_bipartite(G))
    is_planar = bool(nx.check_planarity(G)[0])
    diameter = nx.diameter(G)
    avg_clustering = round(nx.average_clustering(G), 4)
    clique_n = clique_number(G)
    chrom, certified = exact_chromatic_number(G)
    props = {
        "triangle_count": triangle_count,
        "is_bipartite": is_bipartite,
        "is_planar": is_planar,
        "diameter": diameter,
        "chromatic_number": chrom,
        "avg_clustering": avg_clustering,
    }
    extra = {"clique_number": clique_n, "chromatic_number_certified": certified}
    return props, extra


# Step 2: relabel with a random permutation of 0..n-1 (see markdown above).
def randomize_labeling(G, rng):
    nodes = list(G.nodes())
    shuffled = nodes[:]
    rng.shuffle(shuffled)
    mapping = {old: new for old, new in zip(nodes, shuffled)}
    return nx.relabel_nodes(G, mapping, copy=True)


# Step 3: serialize to the edge-list format (Section 4.3).
def to_edge_list_string(G):
    n = G.number_of_nodes()
    m = G.number_of_edges()
    edges = sorted((min(u, v), max(u, v)) for u, v in G.edges())
    lines = [f"GRAPH (n={n}, m={m}):"] + [f"{u} {v}" for u, v in edges]
    return "\n".join(lines)


# Build one full dataset record, enforcing the correct step order.
def build_record(G, tier, family, index, rng, seed, gen_params):
    props, extra = compute_presentation_independent(G)   # step 1
    G2 = randomize_labeling(G, rng)                       # step 2
    edge_list = to_edge_list_string(G2)                   # step 3
    props["degree_of_node_0"] = G2.degree(0)              # step 4
    props["edge_count"] = G2.number_of_edges()

    return {
        "object_id": f"graph_{tier}_{family}_{index:03d}",
        "tier": tier,
        "family": family,
        "num_nodes": G2.number_of_nodes(),
        "num_edges": G2.number_of_edges(),
        "edge_list": edge_list,
        "properties": props,
        "metadata": {
            "generation_params": gen_params,
            "random_seed": seed,
            "clique_number": extra["clique_number"],
            "chromatic_number_certified": extra["chromatic_number_certified"],
            "is_connected": True,
        },
    }


print("Ground-truth and record builder defined (8 properties).")

## 5. Build the 300-graph dataset + summary

300 graphs = 3 tiers x 100. Within each tier: 25 graphs in each of the 4
(bipartite, planar) quadrants (Section 2). Generation failure is **fatal**:
if a generator exhausts its retry budget, `build_dataset` raises rather than
emitting a short dataset.

In [ ]:
def build_dataset(seed=42):
    rng = random.Random(seed)
    records = []
    for tier in ["simple", "medium", "hard"]:
        vmin, vmax = TIERS[tier]

        for i in range(1, PER_QUADRANT + 1):
            G, params = gen_bipartite_planar(rng, vmin, vmax)
            if G is None:
                raise RuntimeError(f"Q1 (bipartite_planar) failed {tier}#{i}")
            records.append(build_record(G, tier, "bipartite_planar", i, rng, seed, params))

        for i in range(1, PER_QUADRANT + 1):
            G, params = gen_bipartite_nonplanar(rng, vmin, vmax)
            if G is None:
                raise RuntimeError(f"Q2 (random_bipartite) failed {tier}#{i}")
            records.append(build_record(G, tier, "random_bipartite", i, rng, seed, params))

        for i in range(1, PER_QUADRANT + 1):
            G, params = gen_nonbipartite_planar(rng, vmin, vmax)
            if G is None:
                raise RuntimeError(f"Q3 (random_planar) failed {tier}#{i}")
            records.append(build_record(G, tier, "random_planar", i, rng, seed, params))

        fam_counts = {"erdos_renyi": 9, "barabasi_albert": 8, "watts_strogatz": 8}
        for fam, cnt in fam_counts.items():
            for i in range(1, cnt + 1):
                G, params = gen_nonbipartite_nonplanar(rng, vmin, vmax, fam)
                if G is None:
                    raise RuntimeError(f"Q4/{fam} failed {tier}#{i}")
                records.append(build_record(G, tier, fam, i, rng, seed, params))

    return records


def stats_for(values):
    n = len(values)
    mean = sum(values) / n
    sv = sorted(values)
    median = sv[n // 2] if n % 2 else (sv[n // 2 - 1] + sv[n // 2]) / 2
    var = sum((v - mean) ** 2 for v in values) / n
    return {"min": round(min(values), 2), "max": round(max(values), 2),
            "mean": round(mean, 2), "median": round(median, 2),
            "std": round(var ** 0.5, 2)}


def summarize(records):
    summary = {"total": len(records)}
    by_tier = {}
    for r in records:
        by_tier.setdefault(r["tier"], {}).setdefault(r["family"], 0)
        by_tier[r["tier"]][r["family"]] += 1
    summary["counts_by_tier_family"] = by_tier

    n_bip = sum(1 for r in records if r["properties"]["is_bipartite"])
    n_planar = sum(1 for r in records if r["properties"]["is_planar"])
    summary["bipartite_overall"] = n_bip
    summary["planar_overall"] = n_planar

    bal = {}
    for tier in ["simple", "medium", "hard"]:
        rs = [r for r in records if r["tier"] == tier]
        b = sum(1 for r in rs if r["properties"]["is_bipartite"])
        p = sum(1 for r in rs if r["properties"]["is_planar"])
        bal[tier] = {"bipartite": b, "not_bipartite": len(rs) - b,
                     "planar": p, "not_planar": len(rs) - p}
    summary["boolean_balance_by_tier"] = bal

    dist = {}
    for tier in ["simple", "medium", "hard"]:
        rs = [r for r in records if r["tier"] == tier]
        dist[tier] = {
            "num_nodes": stats_for([r["num_nodes"] for r in rs]),
            "num_edges": stats_for([r["num_edges"] for r in rs]),
            "triangle_count": stats_for([r["properties"]["triangle_count"] for r in rs]),
            "diameter": stats_for([r["properties"]["diameter"] for r in rs]),
            "chromatic_number": stats_for([r["properties"]["chromatic_number"] for r in rs]),
            "avg_clustering": stats_for([r["properties"]["avg_clustering"] for r in rs]),
            "edge_list_length": stats_for([len(r["edge_list"]) for r in rs]),
        }
    summary["distribution_by_tier"] = dist

    uncertified = [r["object_id"] for r in records if not r["metadata"]["chromatic_number_certified"]]
    summary["chromatic_number_uncertified_count"] = len(uncertified)
    summary["chromatic_number_uncertified_ids"] = uncertified
    return summary


SEED = 42
print("Generating 300-graph dataset (seed =", SEED, ")...")
t0 = time.time()
records = build_dataset(SEED)
print(f"Done in {time.time() - t0:.1f}s.")

with open("graph_exp1_dataset.json", "w") as f:
    json.dump(records, f, indent=2)
print("Saved graph_exp1_dataset.json with", len(records), "graphs.")

summary = summarize(records)
with open("graph_exp1_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("Saved graph_exp1_summary.json.\n")

print("Total graphs:", summary["total"])
print("Counts by tier/family:")
for tier, fams in summary["counts_by_tier_family"].items():
    print(f"  {tier}: {fams}")
print(f"\nBipartite overall: {summary['bipartite_overall']}/300"
      f" ({100*summary['bipartite_overall']/300:.1f}%)")
print(f"Planar overall:    {summary['planar_overall']}/300"
      f" ({100*summary['planar_overall']/300:.1f}%)")
print("\nBoolean balance by tier (should be 50/50 exactly on both):")
for tier, b in summary["boolean_balance_by_tier"].items():
    print(f"  {tier}: bipartite {b['bipartite']}/{b['bipartite']+b['not_bipartite']}, "
          f"planar {b['planar']}/{b['planar']+b['not_planar']}")
print(f"\nChromatic number uncertified: {summary['chromatic_number_uncertified_count']}"
      f" {summary['chromatic_number_uncertified_ids']}")
print("\nNode-count range per tier:")
for tier, d in summary["distribution_by_tier"].items():
    nc = d["num_nodes"]
    print(f"  {tier}: min {nc['min']}, max {nc['max']}, mean {nc['mean']}")

## 6. Independent verification

Re-parse each record **from the stored edge-list string** with
`parse_edge_list` (plain text parsing, no networkx) and recompute properties
with hand-written implementations — BFS-based bipartiteness and diameter,
adjacency-set triangle counting, hand-rolled clustering coefficient — then
compare against the stored ground truth.

**Known gap** (documented, same discipline as the geometry domain's own
gap): this covers 6 of 8 properties. `is_planar` and `chromatic_number` are
not independently re-derived — an independent planarity test (Boyer–Myrvold)
and an independent exact-coloring implementation are both substantial
undertakings on their own; worth closing in a later pass, same as geometry's
unclosed `bbox`/`centroid`/`convex`/`orientation` gap.

In [ ]:
def parse_edge_list(text):
    lines = text.strip().split("\n")
    m = re.match(r"GRAPH \(n=(\d+), m=(\d+)\):", lines[0])
    n_hdr, m_hdr = int(m.group(1)), int(m.group(2))
    edges = []
    for line in lines[1:]:
        line = line.strip()
        if not line:
            continue
        u, v = map(int, line.split())
        edges.append((u, v))
    return n_hdr, m_hdr, edges


def indep_degree0(edges):
    return sum(1 for u, v in edges if u == 0 or v == 0)


def indep_triangle_count(n, edges):
    adj = [set() for _ in range(n)]
    for u, v in edges:
        adj[u].add(v); adj[v].add(u)
    count = 0
    for u in range(n):
        for v in adj[u]:
            if v > u:
                count += sum(1 for w in (adj[u] & adj[v]) if w > v)
    return count


def indep_is_bipartite(n, edges):
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v); adj[v].append(u)
    color = [-1] * n
    for start in range(n):
        if color[start] != -1:
            continue
        color[start] = 0
        q = deque([start])
        while q:
            u = q.popleft()
            for w in adj[u]:
                if color[w] == -1:
                    color[w] = 1 - color[u]
                    q.append(w)
                elif color[w] == color[u]:
                    return False
    return True


def indep_diameter(n, edges):
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v); adj[v].append(u)
    diam = 0
    for start in range(n):
        dist = [-1] * n
        dist[start] = 0
        q = deque([start])
        while q:
            u = q.popleft()
            for w in adj[u]:
                if dist[w] == -1:
                    dist[w] = dist[u] + 1
                    q.append(w)
        if -1 in dist:
            return None    # would indicate a disconnected graph -- shouldn't happen
        diam = max(diam, max(dist))
    return diam


def indep_avg_clustering(n, edges):
    adj = [set() for _ in range(n)]
    for u, v in edges:
        adj[u].add(v); adj[v].add(u)
    total = 0.0
    for u in range(n):
        nb = list(adj[u])
        k = len(nb)
        if k < 2:
            continue
        links = 0
        for i in range(len(nb)):
            for j in range(i + 1, len(nb)):
                if nb[j] in adj[nb[i]]:
                    links += 1
        total += (2 * links) / (k * (k - 1))
    return total / n


mism = 0
for r in records:
    n, m_hdr, edges = parse_edge_list(r["edge_list"])
    gt = r["properties"]
    checks = {
        "num_nodes_header": (n, r["num_nodes"], 0),
        "edge_count_header": (m_hdr, gt["edge_count"], 0),
        "degree_of_node_0": (indep_degree0(edges), gt["degree_of_node_0"], 0),
        "edge_count": (len(edges), gt["edge_count"], 0),
        "triangle_count": (indep_triangle_count(n, edges), gt["triangle_count"], 0),
        "is_bipartite": (indep_is_bipartite(n, edges), gt["is_bipartite"], 0),
        "diameter": (indep_diameter(n, edges), gt["diameter"], 0),
        "avg_clustering": (indep_avg_clustering(n, edges), gt["avg_clustering"], 1e-4),
    }
    for name, (got, exp, tol) in checks.items():
        ok = (got == exp) if isinstance(exp, bool) else (abs(got - exp) <= tol)
        if not ok:
            mism += 1
            if mism <= 10:
                print(f"MISMATCH {r['object_id']} {name}: indep={got} stored={exp}")

print(f"\nIndependent verification: {mism} mismatches across {len(records)} graphs.")

# connectivity / structural sanity
bad = [r["object_id"] for r in records if not r["metadata"]["is_connected"]]
print("Disconnected records:", len(bad))
print("Properties present:", list(records[0]["properties"].keys()))

## 7. Visual spot-check

Plot one graph per (tier x family) — now 6 families x 3 tiers = 18 graphs —
for an eyeball check.

In [ ]:
FAMILIES_FOR_SPOTCHECK = ["bipartite_planar", "random_bipartite", "random_planar",
                          "erdos_renyi", "barabasi_albert", "watts_strogatz"]


def find(tier, family):
    for r in records:
        if r["tier"] == tier and r["family"] == family:
            return r
    return None

fig, axes = plt.subplots(3, 6, figsize=(24, 12))
for row, tier in enumerate(["simple", "medium", "hard"]):
    for col, family in enumerate(FAMILIES_FOR_SPOTCHECK):
        ax = axes[row][col]
        r = find(tier, family)
        n, m, edges = parse_edge_list(r["edge_list"])
        G = nx.Graph(); G.add_nodes_from(range(n)); G.add_edges_from(edges)
        pos = nx.spring_layout(G, seed=42)
        nx.draw(G, pos, ax=ax, node_size=25, width=0.6, node_color="#4C72B0")
        ax.set_title(f"{tier}/{family}\nn={n} m={m} bip={r['properties']['is_bipartite']} "
                     f"plan={r['properties']['is_planar']}", fontsize=8)
plt.tight_layout()
plt.savefig("spotcheck_exp1_graph.png", dpi=110)
plt.show()
print("Saved spotcheck_exp1_graph.png")

## 8. Download (Colab)

Download the dataset, summary, and spot-check figure. (Skip if running
locally — the files are already saved in the working directory.)

In [ ]:
try:
    from google.colab import files
    files.download("graph_exp1_dataset.json")
    files.download("graph_exp1_summary.json")
    files.download("spotcheck_exp1_graph.png")
except Exception as e:
    print("Not on Colab (files saved locally):", e)